# Refactored BBC Categorization Pipeline

This notebook restructures `OLD_bbc_project.ipynb` into modular steps so you can reuse the same pipeline with any combination of article files and category labels.


## 1. Imports And Setup

Run this cell first.


In [ ]:
import json
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

try:
    import tensorflow as tf
    from tensorflow.keras.layers import Dense
    from tensorflow.keras.models import Sequential
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
    tf = None

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")

STOP_WORDS = set(stopwords.words("english"))
RANDOM_STATE = 42


## 2. File Registry

Edit this dictionary whenever you add new datasets.


In [ ]:
AVAILABLE_FILES = {
    "arts": "bbc_arts_articles.json",
    "business": "bbc_business_articles.json",
    "sport": "bbc_sport_articles.json",
}

AVAILABLE_FILES


## 3. Loading Helpers

These functions let you build datasets from as many files and categories as you want.


In [ ]:
def load_json_records(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_article(record, forced_category=None):
    category = (forced_category or record.get("category", "")).strip().lower()
    if not category:
        return None

    return {
        "category": category,
        "title": record.get("title", ""),
        "content": record.get("content", ""),
        "url": record.get("url", ""),
    }


def load_articles_from_source(source):
    path = source["path"]
    forced_category = source.get("category")
    limit = source.get("limit")

    raw_records = load_json_records(path)
    articles = []

    for record in raw_records:
        article = normalize_article(record, forced_category=forced_category)
        if article is not None:
            articles.append(article)

    if limit is not None:
        articles = articles[:limit]

    return articles


def build_sources_from_categories(selected_categories):
    sources = []
    for category in selected_categories:
        if category not in AVAILABLE_FILES:
            raise ValueError(f"Unknown category: {category}. Available categories: {sorted(AVAILABLE_FILES)}")
        sources.append({"path": AVAILABLE_FILES[category], "category": category})
    return sources


def load_dataset(sources):
    dataset = []
    for source in sources:
        dataset.extend(load_articles_from_source(source))
    return dataset


## 4. Preprocessing Helpers

This is the reusable text-cleaning logic extracted from the old notebook.


In [ ]:
def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    cleaned_tokens = []

    for token in tokens:
        if re.fullmatch(r"[a-zA-Z]+", token) and token not in STOP_WORDS:
            cleaned_tokens.append(token)

    return cleaned_tokens


def enrich_articles(articles):
    enriched = []
    for article in articles:
        full_text = f"{article['title']} {article['content']}".strip()
        tokens = preprocess_text(full_text)

        new_article = article.copy()
        new_article["full_text"] = full_text
        new_article["tokens"] = tokens
        enriched.append(new_article)

    return enriched


def summarize_dataset(articles):
    counts = Counter(article["category"] for article in articles)
    print("Articles per category:")
    for category, count in sorted(counts.items()):
        print(f"- {category}: {count}")
    print(f"Total articles: {len(articles)}")


def get_common_tokens(articles, top_n=20):
    token_counts = Counter(token for article in articles for token in article["tokens"])
    common_tokens = token_counts.most_common(top_n)
    for token, freq in common_tokens:
        print(f"{token}: {freq}")
    return common_tokens


def get_common_tokens_by_category(articles, top_n=10):
    grouped_tokens = defaultdict(list)
    for article in articles:
        grouped_tokens[article["category"]].extend(article["tokens"])

    result = {}
    for category, tokens in grouped_tokens.items():
        result[category] = Counter(tokens).most_common(top_n)
        print(f"\nTop tokens for {category}:")
        for token, freq in result[category]:
            print(f"{token}: {freq}")

    return result


## 5. Evaluation Helpers

Shared reporting utilities for every model.


In [ ]:
def show_evaluation(y_true, y_pred, title, labels=None, cmap="Greens"):
    if labels is None:
        labels = sorted(set(y_true) | set(y_pred))

    print(title)
    print(classification_report(y_true, y_pred, labels=labels))
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap=cmap)
    plt.title(title)
    plt.show()


## 6. Dynamic Baseline Classifier

The baseline now learns its buzzwords automatically from the training split by taking the top tokens for each category.


In [ ]:
def count_buzzwords(tokens, buzzwords):
    return sum(token in buzzwords for token in tokens)


def extract_top_tokens_by_category(articles, categories, top_n=10):
    learned_keywords = {}

    for category in categories:
        category_tokens = [
            token
            for article in articles
            if article["category"] == category
            for token in article["tokens"]
        ]
        learned_keywords[category] = [
            token for token, _ in Counter(category_tokens).most_common(top_n)
        ]

    return learned_keywords


def print_learned_keywords(keyword_map):
    print("Learned baseline keywords from the training split:")
    for category, keywords in keyword_map.items():
        print(f"- {category}: {keywords}")


def run_baseline(train_articles, test_articles, categories, top_n=10, keyword_map=None):
    if keyword_map is None:
        keyword_map = extract_top_tokens_by_category(train_articles, categories, top_n=top_n)

    missing = [category for category in categories if category not in keyword_map]
    if missing:
        raise ValueError(f"Missing baseline keywords for categories: {missing}")

    print_learned_keywords(keyword_map)

    y_true = [article["category"] for article in test_articles]
    y_pred = []

    for article in test_articles:
        counts = {
            category: count_buzzwords(article["tokens"], keyword_map[category])
            for category in categories
        }
        prediction = max(counts, key=counts.get)
        y_pred.append(prediction)

    show_evaluation(y_true, y_pred, "Baseline", labels=categories, cmap="Blues")
    return y_pred, keyword_map


## 7. TF-IDF Preparation

This prepares the train/test split and the vectorized features.


In [ ]:
def split_articles(articles, test_size=0.2):
    article_indices = list(range(len(articles)))
    labels = [article["category"] for article in articles]

    train_indices, test_indices = train_test_split(
        article_indices,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=labels,
    )

    train_articles = [articles[idx] for idx in train_indices]
    test_articles = [articles[idx] for idx in test_indices]

    print(f"Training set size: {len(train_articles)}")
    print(f"Test set size: {len(test_articles)}")

    return train_articles, test_articles


def prepare_features(train_articles, test_articles, min_df=5, max_df=100, ngram_range=(1, 2)):
    X_train = [article["full_text"] for article in train_articles]
    X_test = [article["full_text"] for article in test_articles]
    y_train = [article["category"] for article in train_articles]
    y_test = [article["category"] for article in test_articles]

    vectorizer = TfidfVectorizer(min_df=min_df, max_df=max_df, ngram_range=ngram_range)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    print(f"TF-IDF train shape: {X_train_tfidf.shape}")
    print(f"TF-IDF test shape: {X_test_tfidf.shape}")

    return {
        "vectorizer": vectorizer,
        "train_articles": train_articles,
        "test_articles": test_articles,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "X_train_tfidf": X_train_tfidf,
        "X_test_tfidf": X_test_tfidf,
    }


## 8. Logistic Regression

Reusable classical model from the original notebook.


In [ ]:
def run_logistic_regression(prepared_data):
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(prepared_data["X_train_tfidf"], prepared_data["y_train"])

    y_pred = model.predict(prepared_data["X_test_tfidf"])
    show_evaluation(prepared_data["y_test"], y_pred, "Logistic Regression", labels=list(model.classes_), cmap="Greens")

    return model, y_pred


## 9. Neural Network

This version supports both binary and multi-class classification.


In [ ]:
def build_neural_network(input_dim, num_classes):
    model = Sequential()
    model.add(Dense(128, activation="relu", input_shape=(input_dim,)))
    model.add(Dense(64, activation="relu"))

    if num_classes == 2:
        model.add(Dense(1, activation="sigmoid"))
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    else:
        model.add(Dense(num_classes, activation="softmax"))
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    return model


def run_neural_network(prepared_data, epochs=10, batch_size=32):
    if not TENSORFLOW_AVAILABLE:
        raise ImportError("TensorFlow is not installed in this environment.")

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(prepared_data["y_train"])
    y_test_encoded = label_encoder.transform(prepared_data["y_test"])

    X_train_dense = prepared_data["X_train_tfidf"].toarray()
    X_test_dense = prepared_data["X_test_tfidf"].toarray()

    num_classes = len(label_encoder.classes_)
    model = build_neural_network(prepared_data["X_train_tfidf"].shape[1], num_classes)
    model.summary()

    history = model.fit(
        X_train_dense,
        y_train_encoded,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        verbose=1,
    )

    y_pred_raw = model.predict(X_test_dense)

    if num_classes == 2:
        y_pred_encoded = (y_pred_raw > 0.5).astype(int).flatten()
    else:
        y_pred_encoded = y_pred_raw.argmax(axis=1)

    y_pred = label_encoder.inverse_transform(y_pred_encoded)
    show_evaluation(prepared_data["y_test"], y_pred, "Neural Network", labels=list(label_encoder.classes_), cmap="Oranges")

    return model, history, y_pred, label_encoder


## 10. Main Pipeline Runner

This is the one function you can reuse at the end of the notebook for every experiment.


In [ ]:
def run_pipeline(
    sources,
    run_baseline_model=True,
    run_logistic_model=True,
    run_neural_model=False,
    show_dataset_summary=True,
    show_top_tokens=False,
    show_top_tokens_by_category=False,
    top_n_tokens=20,
    baseline_top_n=10,
    baseline_keywords=None,
    tfidf_kwargs=None,
    nn_kwargs=None,
):
    tfidf_kwargs = tfidf_kwargs or {}
    nn_kwargs = nn_kwargs or {}

    articles = load_dataset(sources)
    articles = enrich_articles(articles)
    categories = sorted({article["category"] for article in articles})

    if show_dataset_summary:
        summarize_dataset(articles)

    if show_top_tokens:
        print(f"\nTop {top_n_tokens} tokens in the whole dataset:")
        get_common_tokens(articles, top_n=top_n_tokens)

    if show_top_tokens_by_category:
        get_common_tokens_by_category(articles, top_n=top_n_tokens)

    results = {
        "articles": articles,
        "categories": categories,
        "sources": sources,
    }

    print("\n=== Train/Test Split ===")
    test_size = tfidf_kwargs.pop("test_size", 0.2)
    train_articles, test_articles = split_articles(articles, test_size=test_size)
    results["train_articles"] = train_articles
    results["test_articles"] = test_articles
    results["test_size"] = test_size

    if run_baseline_model:
        print("\n=== Baseline ===")
        baseline_predictions, learned_keywords = run_baseline(
            train_articles,
            test_articles,
            categories,
            top_n=baseline_top_n,
            keyword_map=baseline_keywords,
        )
        results["baseline_predictions"] = baseline_predictions
        results["baseline_keywords"] = learned_keywords

    if run_logistic_model or run_neural_model:
        print("\n=== Feature Preparation ===")
        prepared_data = prepare_features(train_articles, test_articles, **tfidf_kwargs)
        results["prepared_data"] = prepared_data

    if run_logistic_model:
        print("\n=== Logistic Regression ===")
        logistic_model, logistic_predictions = run_logistic_regression(prepared_data)
        results["logistic_model"] = logistic_model
        results["logistic_predictions"] = logistic_predictions

    if run_neural_model:
        print("\n=== Neural Network ===")
        neural_model, neural_history, neural_predictions, label_encoder = run_neural_network(prepared_data, **nn_kwargs)
        results["neural_model"] = neural_model
        results["neural_history"] = neural_history
        results["neural_predictions"] = neural_predictions
        results["label_encoder"] = label_encoder

    return results


## 11. Ready-To-Use Pipelines

Below are examples you can copy and adapt.


### Example A: Arts + Business

Two categories, two files.


In [ ]:
pipeline_arts_business = run_pipeline(
    sources=[
        {"path": AVAILABLE_FILES["arts"], "category": "arts"},
        {"path": AVAILABLE_FILES["business"], "category": "business"},
    ],
    run_baseline_model=True,
    run_logistic_model=True,
    run_neural_model=True,
    show_top_tokens=False,
)


### Example B: Arts + Business + Sport

Three categories, three files.


In [ ]:
pipeline_arts_business_sport = run_pipeline(
    sources=[
        {"path": AVAILABLE_FILES["arts"], "category": "arts"},
        {"path": AVAILABLE_FILES["business"], "category": "business"},
        {"path": AVAILABLE_FILES["sport"], "category": "sport"},
    ],
    run_baseline_model=True,
    run_logistic_model=True,
    run_neural_model=True,
    show_top_tokens=False,
)


### Example C: Build Sources From Category Names

This is useful when your categories are already listed in `AVAILABLE_FILES`.


In [ ]:
selected_categories = ["arts", "sport"]
selected_sources = build_sources_from_categories(selected_categories)

pipeline_from_category_list = run_pipeline(
    sources=selected_sources,
    run_baseline_model=True,
    run_logistic_model=True,
    run_neural_model=True,
)


### Example D: Fully Custom Pipeline Template

Duplicate this cell and change paths, labels, limits, and model options however you want.


In [ ]:
custom_sources = [
    {"path": "bbc_arts_articles.json", "category": "arts"},
    {"path": "bbc_business_articles.json", "category": "business"},
    {"path": "bbc_sport_articles.json", "category": "sport", "limit": 300},
]

custom_pipeline = run_pipeline(
    sources=custom_sources,
    run_baseline_model=True,
    run_logistic_model=True,
    run_neural_model=True,
    show_dataset_summary=True,
    show_top_tokens=True,
    show_top_tokens_by_category=False,
    top_n_tokens=15,
    tfidf_kwargs={
        "test_size": 0.2,
        "min_df": 5,
        "max_df": 100,
        "ngram_range": (1, 2),
    },
    nn_kwargs={
        "epochs": 10,
        "batch_size": 32,
    },
)
